## Sprint 4: Search Optimization — K-Means & IVF Logic

Brute force search is O(n) — every new document makes every future
query slower. The solution: cluster documents offline, then at query
time only search the nearest cluster.

This is exactly how FAISS IVF indexes work internally.
We implement it from scratch using only NumPy.

>
---

**Generating vectors to cluster**
>
We will work in 2D so we can visualize our clusters clearly. In production, this would be 768 normalized embeddings — the math is identical, just more dimensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

group_a = np.random.randn(7, 2) + np.array([4.0, 4.0])
group_b = np.random.randn(7, 2) + np.array([-4.0, 4.0])
group_c = np.random.randn(6, 2) + np.array([0.0, -4.0])

vectors = np.vstack([group_a, group_b, group_c])
print(f"Vector corpus shape: {vectors.shape}")

plt.figure(figsize=(6, 5))
plt.scatter(vectors[:, 0], vectors[:, 1],
            c="steelblue", s=80, alpha=0.7, zorder=3)
plt.title("20 document vectors — before clusterings")
plt.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

**Implementing K-Means from scratch**

K-means clustering implemented from scratch using only NumPy.
>
The algorithm:
1. Pick K random vectors as starting centroids
2. Assign every vector to its nearest centroid
3. Recompute each centroid as the mean of its assigned vectors
4. Repeat 2-3 until centroids stop moving (converge)
>
Args:
- vectors: (n, d) array of document embeddings
- k: number of clusters to form
- max_iters: safety cap on iterations
- tol: converge threshold - stop if centroids move less than this between iterations
>
Returns:
- centroids: (k, d) final centroid positions
- assignments: (n,) cluster index for each vector


In [ ]:
def kmeans(vectors, k, max_iters=100, tol=1e-4):

    n, d = vectors.shape

    random_indices = np.random.choice(n, k, replace=False)
    centroids = vectors[random_indices]

    assignments = np.zeros(n, dtype=int)

    for iteration in range(max_iters):
        distances = np.zeros((n, k))
        for j, centroid in enumerate(centroids):
            diff = vectors - centroid
            distances[:, j] = np.sqrt(np.sum(diff**2, axis=1))

        new_assignments = np.argmin(distances, axis=1)

        new_centroids = np.zeros_like(centroids)
        for j in range(k):
            members = vectors[new_assignments == j]
            if len(members) > 0:
                new_centroids[j] = np.mean(members, axis=0)
            else: 
                new_centroids[j] = centroids[j]

        centroid_shift = np.sqrt(np.sum((new_centroids - centroids) ** 2))
        assignments = new_assignments
        centroids = new_centroids

        print(f"Iteration {iteration + 1:02d}: centroid shifts = {centroid_shift:.6f}")

        if centroid_shift < tol:
            print(f"\nConverged after {iteration + 1} iterations ✓")
            break
    return centroids, assignments

# Run K-Means with K=3

np.random.seed(0)
K = 3
centroids, assignments = kmeans(vectors, K)

print(f"\nFinal Centroids: \n{np.round(centroids, 3)}")
print(f"Assignments: {assignments}")

**Visualize the Clusters**

In [ ]:
colors  = ["#1D9E75", "#534AB7", "#E24B4A"]
labels  = ["Cluster 0 (animals)", "Cluster 1 (tech)", "Cluster 2 (finance)"]
markers = ["o", "s", "^"]

fig, ax = plt.subplots(figsize=(7, 6))

for j in range(K):
    members = vectors[assignments == j]
    ax.scatter(members[:, 0], members[:, 1],
               color=colors[j], label=labels[j],
               s=90, alpha=0.75, zorder=3, marker=markers[j])
    ax.scatter(centroids[j, 0], centroids[j, 1],
               color=colors[j], s=280, marker="*",
               edgecolors="black", linewidths=1, zorder=5)
    ax.annotate(f"μ{j+1}",
                xy=(centroids[j, 0], centroids[j, 1]),
                xytext=(8, 6), textcoords="offset points",
                fontsize=11, fontweight="bold", color=colors[j])

ax.set_title("K-Means Clustering (K=3)\nimplemented from scratch — no Scikit-learn",
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig("../images/kmeans_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

**Implement IVF-style Search**

IVF-style approximate nearest neighbour search.
>
Instead of comparing query against all n vectors, we first find the nearest centroid (cheap), then only search within that cluster (much smaller set).
>
This is exactly the logic inside FAISS IVF indexes.
>
Args:
- query:       (d,)  query vector
- vectors:     (n, d) all document vectors
- centroids:   (k, d) cluster centroids from K-Means
- assignments: (n,)  cluster membership for each vector
- top_k:       how many results to return
>
Returns:
- top_k indices and their distances to the query



In [ ]:
def ivf_search(query, vectors, centroids, assignments, top_k=3):

# Find nearest centroid
    centroid_distances = np.sqrt(
        np.sum((centroids - query) ** 2, axis=1)
    )
    nearest_cluster = np.argmin(centroid_distances)

    print(f"Query routed to cluster {nearest_cluster}")
    print(f"Centroid distances: {np.round(centroid_distances, 3)}")

# Search within that cluster only
    cluster_mask = assignments == nearest_cluster
    cluster_indices = np.where(cluster_mask)[0]
    cluster_vectors = vectors[cluster_mask]

    print(f"Searching {len(cluster_indices)} vectors "
          f"instead of {len(vectors)} — "
          f"{len(cluster_indices)/len(vectors)*100:.0f}% of corpus\n")
    
    # Euclidean distance within cluster
    distances = np.sqrt(
        np.sum((cluster_vectors - query) ** 2, axis=1)
    )

    # Return top_k nearest within that cluster
    top_k_local = np.argsort(distances)[:top_k]
    top_k_global = cluster_indices[top_k_local]

    return top_k_global, distances[top_k_local]

# Test witgh a query near the tech cluster
query = np.array([-3.8, 4.2])

print("=== IVF Search ===")
result_indices, result_distances = ivf_search(
    query, vectors, centroids, assignments, top_k=3
)

print("Top-3 nearest neighbours (IVF):")
for rank, (idx, dist) in enumerate(zip(result_indices, result_distances)):
    print(f"  Rank {rank+1}: vector[{idx}] = {np.round(vectors[idx], 3)}"
          f"  distance = {dist:.4f}")


**Prove IVF agrees with Brute Force**

O(n) search — compare query against every single vector.
Ground truth for verifying IVF correctness.

In [ ]:
def brute_force_search(query, vectors, top_k=3):
    
    distances = np.sqrt(np.sum((vectors - query) ** 2, axis=1))
    top_k_indices = np.argsort(distances)[:top_k]
    return top_k_indices, distances[top_k_indices]

bf_indices, bf_distances = brute_force_search(query, vectors, top_k=3)

print("=== Brute Force Search (ground truth) ===")
print("Top-3 nearest neighbours:")
for rank, (idx, dist) in enumerate(zip(bf_indices, bf_distances)):
    print(f"  Rank {rank+1}: vector[{idx}] = {np.round(vectors[idx], 3)}"
          f"  distance = {dist:.4f}")

print()
print("=== Agreement Check ===")
ivf_set = set(result_indices)
bf_set  = set(bf_indices)
overlap = len(ivf_set & bf_set)
print(f"IVF found:         {sorted(ivf_set)}")
print(f"Brute force found: {sorted(bf_set)}")
print(f"Overlap: {overlap}/3 results agree")

if overlap == 3:
    print("✓ IVF matches brute force exactly on this query")
else:
    print(f"IVF approximation: missed {3 - overlap} result(s)")
    print("This is expected — IVF trades perfect recall for speed")

# Sprint 4: Search Optimization — K-Means & IVF

## The Problem
Brute force search is O(n). At 1 million documents, comparing a query
against every vector is computationally impossible at production scale.

## The Solution: Cluster Offline, Route at Query Time
Instead of searching everything, we:
1. **Cluster** all vectors into K neighbourhoods using K-Means (done once, offline)
2. **Route** the query to the nearest cluster centroid (K comparisons, essentially free)
3. **Search** only within that cluster (~n/K vectors instead of n)

## K-Means: How It Works
Two steps, repeated until centroids stop moving:

$$J = \sum_{k=1}^{K} \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$$

- **Assignment** — assign every vector to its nearest centroid
- **Update** — recompute each centroid as the mean of its members

## IVF Search: Two-Phase Lookup
$$\text{Phase 1: } c^* = \arg\min_{k} \|\mathbf{q} - \boldsymbol{\mu}_k\|^2
\quad \text{(route query to nearest centroid)}$$

$$\text{Phase 2: search vectors where } \text{assignment} = c^* \text{ only}$$

## The Tradeoff
| Approach | Comparisons | Recall |
|---|---|---|
| Brute force | n | 100% |
| IVF (nprobe=1) | K + n/K | ~90-95% |

IVF is an **approximation** — queries near cluster boundaries can
miss their true nearest neighbour. Production systems tune `nprobe`
(number of clusters searched) to control the speed vs recall tradeoff.

## What This Maps To in Production
- `kmeans()` → FAISS IVF index build
- `ivf_search()` → FAISS `IndexIVFFlat.search()`
- `nprobe` → how many clusters to search per query
